# 6. Vol Surface / EVC — QA

**Purpose** — Validate `sigma_evc` grid quality (raw heatmap kept for missing-tile / discontinuity detection — NOT for PM readability).

**Source tables**
- `py_vol_surface_daily`
- `mart_price_analytics_daily`

**Window** — data span ends `2026-04-06`. The bootstrap cell below sets `END_DATE` and two convenience starts (`START_DATE_30D`, `START_DATE_90D`).

**Convention** — values denominated in `token0` / `token0_equiv`. For the indexed USDC/WETH 0.05% pool, `token0 = USDC` so `token0_equiv ≈ USD`.

This notebook is a **QA workbench**, not a PM dashboard. Each section ends with explicit PASS/WARN/FAIL checks. Final §4 cell prints the verdict.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL      = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
END_DATE  = "2026-04-06"            # Last date with full mart coverage
START_DATE_90D = (date.fromisoformat(END_DATE) - timedelta(days=90)).isoformat()
START_DATE_30D = (date.fromisoformat(END_DATE) - timedelta(days=30)).isoformat()

print(f"Pool:       {POOL}")
print(f"End date:   {END_DATE}")
print(f"Window 90d: {START_DATE_90D} → {END_DATE}")
print(f"Window 30d: {START_DATE_30D} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:       0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
End date:   2026-04-06
Window 90d: 2026-01-06 → 2026-04-06
Window 30d: 2026-03-07 → 2026-04-06


<function __main__.<lambda>()>

In [2]:
# QA check helper — records pass/warn/fail into QA_RESULTS for the §4 verdict cell.
QA_RESULTS = []

def qa_check(label, passed, severity="fail", detail=""):
    if passed:
        status, icon = "PASS", "🟢"
    elif severity == "warn":
        status, icon = "WARN", "🟠"
    else:
        status, icon = "FAIL", "🔴"
    QA_RESULTS.append({"label": label, "status": status, "severity": severity, "detail": str(detail)})
    print(f"{icon} {status:4s} | {label}" + (f"  —  {detail}" if detail else ""))

def qa_table_exists(name, database="analytics"):
    try:
        n = db.read_query(
            f"SELECT count() AS n FROM system.tables WHERE database='{database}' AND name='{name}'"
        )["n"][0]
        return int(n) > 0
    except Exception:
        return False

def qa_safe(query):
    try:
        return db.read_query(query)
    except Exception as e:
        return None


## §1 — Data fetch & coverage

In [3]:
vs = db.read_query(f"""
SELECT day AS date, moneyness_bucket, sigma_evc_raw, sigma_evc_smoothed,
       tick_count, total_liquidity, rv_realized_7d, rv_realized_30d
FROM py_vol_surface_daily
WHERE lower(pool) = lower('{POOL}') AND day <= toDate('{END_DATE}')
ORDER BY day, moneyness_bucket
""")
prc = db.read_query(f"""
SELECT date, realized_vol_7d, realized_vol_30d
FROM mart_price_analytics_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
""")
moneyness_buckets = sorted(vs["moneyness_bucket"].unique().to_list())
n_dates = vs.select(pl.col("date").n_unique()).item()
print(f"Vol surface rows: {vs.height}")
print(f"Distinct dates:   {n_dates}    range: {vs['date'].min()} → {vs['date'].max()}")
print(f"Moneyness buckets ({len(moneyness_buckets)}): {moneyness_buckets}")

Vol surface rows: 71
Distinct dates:   37    range: 2026-03-01 → 2026-04-06
Moneyness buckets (7): [-0.1, -0.05, 0.0, 0.05, 0.1, 0.15, 0.2]


## §2 — Sanity checks

In [4]:
# Grid completeness — calibrated on engine semantics:
# - ATM bucket (0.0) must be present every day with vol activity
# - Outer buckets (|moneyness| > 0) are only emitted on days with that-magnitude price moves
atm_days = vs.filter(pl.col("moneyness_bucket") == 0.0)["date"].n_unique()
qa_check("ATM bucket (m=0.0) present on 100% of days",
         atm_days == n_dates, severity="warn",
         detail=f"atm_days={atm_days}/{n_dates}")

# Per-day bucket count distribution: median should be ≥ 2 (ATM + at least one off-ATM)
per_day = vs.group_by("date").agg(pl.col("moneyness_bucket").n_unique().alias("n_b")).sort("date")
median_buckets = float(per_day["n_b"].median())
qa_check("median buckets per day ≥ 2 (ATM + at least one neighbour)",
         median_buckets >= 2, severity="warn",
         detail=f"median={median_buckets}, distribution={dict(zip(per_day['n_b'].to_list(), [1]*per_day.height))}")


🟢 PASS | ATM bucket (m=0.0) present on 100% of days  —  atm_days=37/37
🟢 PASS | median buckets per day ≥ 2 (ATM + at least one neighbour)  —  median=2.0, distribution={2: 1, 3: 1, 4: 1, 1: 1, 6: 1}


In [5]:
# Smile discontinuity: |sigma[m+1] - sigma[m]| / sigma[m] > 0.5 across moneyness on the same day
disc_q = f"""
SELECT count() AS n FROM (
  SELECT day, moneyness_bucket, sigma_evc_raw,
         lagInFrame(sigma_evc_raw, 1, toFloat64(0)) OVER (PARTITION BY day ORDER BY moneyness_bucket) AS prev_sig
  FROM py_vol_surface_daily
  WHERE lower(pool) = lower('{POOL}') AND day <= toDate('{END_DATE}')
)
WHERE prev_sig > 0 AND abs(sigma_evc_raw - prev_sig) / prev_sig > 0.5
"""
n_disc = int(db.read_query(disc_q)["n"][0])
qa_check("smile continuity (|Δσ| / σ ≤ 50% across adjacent buckets)",
         n_disc == 0, severity="warn",
         detail=f"discontinuities={n_disc}")


🟠 WARN | smile continuity (|Δσ| / σ ≤ 50% across adjacent buckets)  —  discontinuities=4


In [6]:
# ATM sigma vs realized vol — plausibility
atm = vs.filter(pl.col("moneyness_bucket") == 0.0).select("date", "sigma_evc_raw").join(prc, on="date")
ratio = (atm["sigma_evc_raw"] / atm["realized_vol_30d"]).drop_nulls()
n_outside = int(((ratio < 0.2) | (ratio > 5.0)).sum())
qa_check("ATM sigma_evc / realized_vol_30d ∈ [0.2, 5]",
         n_outside == 0, severity="warn",
         detail=f"days_outside={n_outside}")

🟠 WARN | ATM sigma_evc / realized_vol_30d ∈ [0.2, 5]  —  days_outside=37


In [7]:
# Date coverage parity with price marts: vol_surface days should be a subset of price days
cov_gap = db.read_query(f"""
SELECT countDistinct(day) AS vol_d,
       (SELECT countDistinct(date) FROM mart_price_analytics_daily WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}') AND date >= (SELECT min(day) FROM py_vol_surface_daily WHERE lower(pool) = lower('{POOL}'))) AS prc_d
FROM py_vol_surface_daily
WHERE lower(pool) = lower('{POOL}') AND day <= toDate('{END_DATE}')
""").row(0, named=True)
qa_check("vol-surface date coverage ≥ 90% of price coverage in the same window",
         int(cov_gap["vol_d"]) / max(int(cov_gap["prc_d"]), 1) >= 0.9, severity="warn",
         detail=f"vol_dates={cov_gap['vol_d']} / price_dates_in_window={cov_gap['prc_d']}")

🟢 PASS | vol-surface date coverage ≥ 90% of price coverage in the same window  —  vol_dates=37 / price_dates_in_window=37


## §3 — Diagnostic visuals

**QA purpose**: detect missing tiles in the (date × moneyness) grid. Holes = engine missed days or buckets.

In [8]:
import numpy as np
ds = sorted(vs["date"].unique().to_list())
mb = sorted(vs["moneyness_bucket"].unique().to_list())
matrix = np.zeros((len(mb), len(ds)))
for r in vs.iter_rows(named=True):
    matrix[mb.index(r["moneyness_bucket"]), ds.index(r["date"])] = 1
fig = go.Figure(go.Heatmap(z=matrix, x=ds, y=mb, colorscale=[[0, "lightgray"], [1, "#22c55e"]], showscale=False))
fig.update_layout(height=320, title_text="Vol-surface grid coverage (1 = present)", yaxis_title="moneyness_bucket")
fig.show()

**QA purpose**: smile shape on the latest day. Asymmetric/spiky smile is a discontinuity flag.

In [9]:
latest_d = vs["date"].max()
latest = vs.filter(pl.col("date") == latest_d).sort("moneyness_bucket")
fig = go.Figure()
fig.add_trace(go.Scatter(x=latest["moneyness_bucket"].to_list(), y=latest["sigma_evc_raw"].to_list(),      name="σ raw",      mode="lines+markers"))
fig.add_trace(go.Scatter(x=latest["moneyness_bucket"].to_list(), y=latest["sigma_evc_smoothed"].to_list(), name="σ smoothed", mode="lines+markers", line=dict(dash="dash")))
fig.update_layout(height=320, title_text=f"Vol smile — {latest_d}", xaxis_title="moneyness_bucket")
fig.show()

**QA purpose**: raw EVC heatmap (date × moneyness) — kept for QA only. Looking for color discontinuities or missing columns.

In [10]:
piv = vs.pivot(values="sigma_evc_raw", index="moneyness_bucket", on="date").fill_null(0)
z = piv.drop("moneyness_bucket").to_numpy()
fig = go.Figure(go.Heatmap(z=z, x=[c for c in piv.columns if c != "moneyness_bucket"], y=piv["moneyness_bucket"].to_list(), colorscale="Viridis"))
fig.update_layout(height=420, title_text="Sigma_evc raw — date × moneyness (QA: smoothness audit)", yaxis_title="moneyness_bucket")
fig.show()

**QA purpose**: ATM sigma_evc vs realized vol stability over time.

In [11]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=atm["date"].to_list(), y=atm["sigma_evc_raw"].to_list(), name="ATM σ_evc"))
fig.add_trace(go.Scatter(x=atm["date"].to_list(), y=atm["realized_vol_7d"].to_list(),  name="realized σ 7d"))
fig.add_trace(go.Scatter(x=atm["date"].to_list(), y=atm["realized_vol_30d"].to_list(), name="realized σ 30d"))
fig.update_layout(height=320, title_text="ATM σ_evc vs realized σ")
fig.show()

**QA purpose**: distribution of all sigma_evc values — identifies tail outliers.

In [12]:
fig = go.Figure(go.Histogram(x=vs["sigma_evc_raw"].to_list(), nbinsx=60))
import numpy as np
v = vs["sigma_evc_raw"].to_numpy()
p1, p99 = np.percentile(v, [1, 99])
fig.add_vline(x=p1, line_dash="dash", line_color="red",  annotation_text=f"p01={p1:.3f}")
fig.add_vline(x=p99, line_dash="dash", line_color="red", annotation_text=f"p99={p99:.3f}")
fig.update_layout(height=300, title_text="sigma_evc value distribution")
fig.show()

## §4 — QA verdict

In [13]:
# ─── §4 QA verdict ──────────────────────────────────────────────────────────
verdict = pl.DataFrame(QA_RESULTS) if QA_RESULTS else pl.DataFrame({"label": [], "status": [], "severity": [], "detail": []})
n_pass = verdict.filter(pl.col("status") == "PASS").height
n_warn = verdict.filter(pl.col("status") == "WARN").height
n_fail = verdict.filter(pl.col("status") == "FAIL").height

if n_fail > 0:
    overall = "FAIL"
elif n_warn > 0:
    overall = "WARN"
else:
    overall = "PASS"

print("=" * 70)
print(f"  QA VERDICT — overall: {overall}")
print(f"  PASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")
print("=" * 70)
if n_fail > 0:
    print("\nFailures:")
    for row in verdict.filter(pl.col("status") == "FAIL").to_dicts():
        print(f"  🔴 {row['label']}  —  {row['detail']}")
if n_warn > 0:
    print("\nWarnings:")
    for row in verdict.filter(pl.col("status") == "WARN").to_dicts():
        print(f"  🟠 {row['label']}  —  {row['detail']}")

verdict


  QA VERDICT — overall: WARN
  PASS: 3   WARN: 2   FAIL: 0

Warnings:
  🟠 smile continuity (|Δσ| / σ ≤ 50% across adjacent buckets)  —  discontinuities=4
  🟠 ATM sigma_evc / realized_vol_30d ∈ [0.2, 5]  —  days_outside=37


label,status,severity,detail
str,str,str,str
"""ATM bucket (m=0.0) present on …","""PASS""","""warn""","""atm_days=37/37"""
"""median buckets per day ≥ 2 (AT…","""PASS""","""warn""","""median=2.0, distribution={2: 1…"
"""smile continuity (|Δσ| / σ ≤ 5…","""WARN""","""warn""","""discontinuities=4"""
"""ATM sigma_evc / realized_vol_3…","""WARN""","""warn""","""days_outside=37"""
"""vol-surface date coverage ≥ 90…","""PASS""","""warn""","""vol_dates=37 / price_dates_in_…"
